## Spark ML query search

This code will be used to compare to the pandas + spark pipeline for semantic search.

Comparing the following:

1. semantic vs keyword
2. Pandas and spark vs spark
3. pandas and spark vs pandas

### Connecting to data

We will be using the RDS movie soup that is the same as the soup for the semantic search.

Connection to AWS RDS:

** Include the secret key to connect **


In [ ]:
from pyspark.sql import SparkSession
from google.colab import userdata

# 1. Initialize Spark with the Postgres Connector
spark = SparkSession.builder \
    .appName("VibeLens_RDS_Connector") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.6.0") \
    .getOrCreate()

# 2. Build the Connection URL and Properties
jdbc_url = f"jdbc:postgresql://{userdata.get('PG_HOST')}:5432/{userdata.get('PG_DATABASE')}"
connection_properties = {
    "user": userdata.get('PG_USER'),
    "password": userdata.get('PG_PASSWORD'),
    "driver": "org.postgresql.Driver",
    "ssl": "true",
    "sslmode": "require"
}

# Query the Postgres 'information_schema' to find your tables
table_list_df = spark.read.jdbc(
    url=jdbc_url,
    table="(SELECT table_name FROM information_schema.tables WHERE table_schema = 'public') as tables",
    properties=connection_properties
)

print("📂 Tables found in your RDS 'public' schema:")
table_list_df.show()

📂 Tables found in your RDS 'public' schema:
+-----------+
| table_name|
+-----------+
|query_cache|
|     movies|
+-----------+



Ok we have successful connection, now to load all the data, and prepare for TFIDF

In [ ]:
# 1. Load the 'movies' table from AWS RDS
movies_df = spark.read.jdbc(url=jdbc_url, table="movies", properties=connection_properties)

print(f"✅ Successfully connected to RDS! Total movies found: {movies_df.count()}")
movies_df.show(5)

# 2. Ensure we have the 'soup' column (Text to be analyzed)
from pyspark.sql.functions import concat_ws, col, lit, coalesce

print(f"✅ Loaded {movies_df.count()} movies from RDS for Spark ML.")

✅ Successfully connected to RDS! Total movies found: 11937
+-------+--------------------+----+--------------------+--------------------+-----------+----------+-----------+--------------------+--------------------+
|movieId|               title|year|              genres|         tmdb_genres|num_ratings|avg_rating|tmdb_rating|                soup|           embedding|
+-------+--------------------+----+--------------------+--------------------+-----------+----------+-----------+--------------------+--------------------+
|   3670|   Story of G.I. Joe|1945|                 War|          Drama, War|        108| 3.3842592|      6.474|Story of G.I. Joe...|[-0.058036946,-0....|
|  93842|Delicacy (La déli...|2011|      Comedy|Romance|Drama, Comedy, Ro...|        102| 3.4656863|      6.343|Delicacy (La déli...|[-0.0036516679,-0...|
|  45501|       Break-Up, The|2006|Comedy|Drama|Romance|     Romance, Comedy|       2946| 2.8791583|      5.902|Break-Up, The is ...|[-0.11687993,-0.0...|
|   1690| A

### TF-IDF Search

Search for moives based on keywords in the search query.

This step takes the words in the soup and tokenizes them based on importance.

Matching these keywords from query to the "movie soup" entries

In [ ]:
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF
from pyspark.ml import Pipeline

# 1. Tokenizer: Split text into words
tokenizer = Tokenizer(inputCol="soup", outputCol="words")

# 2. StopWords: Remove "the", "a", "is" (noise)
remover = StopWordsRemover(inputCol="words", outputCol="filtered_words")

# 3. HashingTF: Convert words to numeric frequencies
hashingTF = HashingTF(inputCol="filtered_words", outputCol="rawFeatures", numFeatures=10000)

# 4. IDF: Calculate the 'Inverse Document Frequency' (Unique words get higher weights)
idf = IDF(inputCol="rawFeatures", outputCol="features")

# 5. Build & Run the Pipeline
pipeline = Pipeline(stages=[tokenizer, remover, hashingTF, idf])
spark_ml_model = pipeline.fit(movies_df)
tfidf_df = spark_ml_model.transform(movies_df)

print("✨ Spark ML TF-IDF features generated successfully!")
tfidf_df.select("title", "features").show(5, truncate=50)

✨ Spark ML TF-IDF features generated successfully!
+-------------------------+--------------------------------------------------+
|                    title|                                          features|
+-------------------------+--------------------------------------------------+
|        Story of G.I. Joe|(10000,[70,350,407,524,696,698,942,970,974,1016...|
|Delicacy (La délicatesse)|(10000,[137,442,775,813,851,1069,1180,1288,1299...|
|            Break-Up, The|(10000,[12,139,159,232,260,283,345,444,447,454,...|
|      Alien: Resurrection|(10000,[209,300,325,326,404,468,585,649,662,759...|
|            Covenant, The|(10000,[23,51,55,60,62,139,249,391,479,567,739,...|
+-------------------------+--------------------------------------------------+
only showing top 5 rows


In [ ]:
from pyspark.sql.functions import udf, col, array
from pyspark.sql.types import FloatType
from pyspark.ml.linalg import VectorUDT, Vectors

# 1. The UDF remains the same
def calculate_cosine_similarity(vec1, vec2):
    if vec1 is not None and vec2 is not None:
        # Check if vectors are non-zero to avoid division by zero
        denom = (vec1.norm(2) * vec2.norm(2))
        return float(vec1.dot(vec2) / denom) if denom != 0 else 0.0
    return 0.0

cosine_sim_udf = udf(calculate_cosine_similarity, FloatType())

def search_spark_ml(query_text, top_k=5):
    # A. Transform the query text
    query_df = spark.createDataFrame([(query_text,)], ["soup"])
    query_vector = spark_ml_model.transform(query_df).select("features").collect()[0][0]

    # B. THE FIX: Define a helper to apply the query vector row-by-row
    # We use a lambda to "bake" the query_vector into the function call
    calculate_sim_with_query = udf(lambda x: calculate_cosine_similarity(x, query_vector), FloatType())

    # C. Apply and Sort
    print(f"\n Spark ML (Keyword) Results for: '{query_text}'")
    print("-" * 50)

    results = tfidf_df.withColumn("score", calculate_sim_with_query(col("features"))) \
                      .select("title", "score") \
                      .orderBy(col("score").desc()) \
                      .limit(top_k)

    results.show()

# --- RUN THE TEST AGAIN ---
search_spark_ml("scary alien on a spaceship")


 Spark ML (Keyword) Results for: 'scary alien on a spaceship'
--------------------------------------------------
+--------------+----------+
|         title|     score|
+--------------+----------+
| Scary Movie 3| 0.2175974|
|         Alien|0.21485725|
|Beyond Skyline|0.16784628|
| Scary Movie 2| 0.1586595|
|            65| 0.1545588|
+--------------+----------+

